# Colebrookova jednadžba: predvidi → izračunaj → provjeri

**Poglavlje 13 — Gubitci, cjevovodi, crpke i mreže**

Moodyjev dijagram nije samo slika: u turbulentnom području svaka njegova krivulja zadovoljava implicitnu Colebrookovu jednadžbu za Darcyjev koeficijent trenja \(\lambda\).

## 1. Predvidi

Prije iteracije procijeni:

1. Povećava li relativna hrapavost \(k_s/D\) koeficijent trenja?
2. Hoće li dvije razumne početne pretpostavke završiti na istom korijenu?
3. Zašto Colebrookovu jednadžbu ne treba primjenjivati na laminarni tok?

Prijelazno područje \(2300\lesssim Re\lesssim4000\) ovdje namjerno ne interpoliramo: režim može biti nestabilan i jedna glatka formula skriva tu neizvjesnost.

Nastavak povezuje Colebrookov račun s radnom točkom iz **Z4 poglavlja 13**: crpka uz jezero puni vodospremu naselja. Predvidi kako će veća hrapavost pomaknuti presjecište karakteristika crpke i sustava. Karakteristika crpke nastavni je model, a ne krivulja određene komercijalne crpke.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 110, "font.size": 10})

def colebrook_residual(lam, Re, rel_roughness):
    return 1/np.sqrt(lam) + 2*np.log10(rel_roughness/3.7 + 2.51/(Re*np.sqrt(lam)))

def colebrook_iter(Re, rel_roughness, x0=7.0, tol=1e-12, max_iter=100):
    # Fiksna točka u varijabli x=1/sqrt(lambda), uz povijest iteracija.
    if Re <= 4000 or rel_roughness < 0:
        raise ValueError("Colebrookov račun ovdje vrijedi za Re > 4000 i k_s/D ≥ 0.")
    x = float(x0)
    history = []
    for iteration in range(max_iter + 1):
        lam = 1/x**2
        residual = colebrook_residual(lam, Re, rel_roughness)
        history.append((iteration, lam, residual))
        x_new = -2*np.log10(rel_roughness/3.7 + 2.51*x/Re)
        if abs(x_new-x) < tol:
            x = x_new
            lam = 1/x**2
            history.append((iteration+1, lam, colebrook_residual(lam, Re, rel_roughness)))
            return lam, np.asarray(history)
        x = x_new
    raise RuntimeError("Iteracija nije konvergirala unutar zadanog broja koraka.")

Re0, rr0 = 1.0e5, 1.0e-4
lam0, history = colebrook_iter(Re0, rr0, x0=5.0)
print(f"lambda = {lam0:.8f}; iteracija = {len(history)-1}; završni rezidual = {history[-1,2]:.3e}")
print(" i       lambda       rezidual")
for row in history:
    print(f"{int(row[0]):2d}   {row[1]:.9f}   {row[2]: .3e}")


## 2. Izračunaj: konvergencija i osjetljivost na hrapavost

Rezidual je lijeva strana implicitne jednadžbe i mora težiti nuli. Zatim za isti Reynoldsov broj mijenjamo \(k_s/D\) kroz četiri reda veličine. To je numerička verzija vodoravnog presjeka Moodyjeva dijagrama.


In [ ]:
lam_other_start, history_other = colebrook_iter(Re0, rr0, x0=12.0)
roughness = np.logspace(-6, -2, 70)
lam_rough = np.array([colebrook_iter(Re0, rr)[0] for rr in roughness])

def friction_factor(Re, rel_roughness=0.0):
    if Re < 2300:
        return 64/Re
    if Re <= 4000:
        raise ValueError("Prijelazno područje nema jedinstvenu vrijednost u ovom modelu.")
    return colebrook_iter(Re, rel_roughness)[0]

lam_laminar = friction_factor(1200, rr0)
print(f"Laminarno, Re=1200: lambda = {lam_laminar:.6f}")
print(f"Turbulentno: promjena λ od {lam_rough[0]:.5f} do {lam_rough[-1]:.5f}")


## 3. Provjeri

Neovisne provjere su: zatvaranje implicitne jednadžbe, neovisnost korijena o početnoj pretpostavci i analitički laminarni granični slučaj \(\lambda=64/Re\).


In [ ]:
assert abs(colebrook_residual(lam0, Re0, rr0)) < 1e-10
assert np.isclose(lam_other_start, lam0, rtol=1e-11)
assert np.isclose(lam_laminar, 64/1200, rtol=1e-14)
assert np.all(np.diff(lam_rough) > 0)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].semilogy(history[:,0], np.maximum(np.abs(history[:,2]), 1e-16), "o-", color="#b43c35")
axes[0].set(xlabel="iteracija", ylabel="|Colebrookov rezidual|", title="Povijest konvergencije")
axes[1].semilogx(roughness, lam_rough, color="#256d85", lw=2)
axes[1].scatter([rr0], [lam0], color="#b43c35", zorder=3)
axes[1].set(xlabel=r"relativna hrapavost $k_s/D$", ylabel=r"Darcyjev $\lambda$", title=rf"Osjetljivost pri $Re={Re0/10**np.floor(np.log10(Re0)):g}\cdot10^{{{np.floor(np.log10(Re0)):.0f}}}$")
for ax in axes: ax.grid(True, which="both", ls=":", alpha=.45)
plt.tight_layout(); plt.show()


## 4. Z4 poglavlja 13: crpka puni vodospremu naselja

Crpka uz jezero ima karakteristiku $H_p=46-9000Q^2$, uz $Q$ u m³/s i visinu u metrima. Razina vode u vodospremi je $Δz=28$ m iznad razine jezera. Tlačni vod ima $D=0{,}150$ m, $L=620$ m, $k_s=0{,}10$ mm i $Σξ=9$, uključujući usisni koš i izlaz u vodospremu. Koeficijenti se odnose na brzinu u vodu. Za vodu vrijedi $ρ=1000$ kg/m³ i $ν=10^{-6}$ m²/s; učinkovitost crpke je 0,74, a motora 0,93. Naselje dnevno troši 1500 m³.

U zadatku se radna točka traži ručno: iz $λ$ slijedi $Q$, iz $Q$ novi $λ$ Swamee–Jainovom relacijom. Ovdje isti problem rješavamo drukčije. Za svaku probnu vrijednost protoka izračunavamo $Re$ i rješavamo Colebrookovu jednadžbu, a vanjska bisekcija zatvara $H_p-H_s=0$. Provjeravamo oba reziduala, konačni turbulentni režim i podudaranje s ručnim rezultatom.

In [ ]:
D4, L4, eps4, xi4 = 0.150, 620.0, 0.10e-3, 9.0
nu4, rho4, g4, dz4 = 1e-6, 1000.0, 9.81, 28.0
eta_p4, eta_m4, V_day4 = 0.74, 0.93, 1500.0
def pump_state4(Q, epsilon=eps4):
    v = 4*Q/(np.pi*D4**2)
    Re = v*D4/nu4
    lam, _ = colebrook_iter(Re, epsilon/D4)
    Hp = 46-9000*Q**2
    Hs = dz4+(lam*L4/D4+xi4)*v**2/(2*g4)
    return Hp, Hs, Re, lam

def operating_point4(epsilon=eps4, tolerance=1e-10):
    lo, hi = 0.005, 0.060
    alo, blo, *_ = pump_state4(lo, epsilon)
    ahi, bhi, *_ = pump_state4(hi, epsilon)
    if not alo>blo or not ahi<bhi:
        raise ValueError("Interval ne obuhvaća radnu točku.")
    history = []
    for iteration in range(80):
        Q = (lo+hi)/2
        Hp, Hs, Re, lam = pump_state4(Q, epsilon)
        residual = Hp-Hs
        history.append((iteration,Q,residual))
        if abs(residual)<tolerance:
            return Q, Hp, Re, lam, np.array(history)
        if residual>0:lo=Q
        else:hi=Q
    raise RuntimeError("Radna točka nije konvergirala.")

Q4, H4, Re4, lam4, history4 = operating_point4()
Ph4 = rho4*g4*Q4*H4
Pvr4 = Ph4/eta_p4
Pel4 = Pvr4/eta_m4
hours4 = V_day4/Q4/3600
print(f"Q = {Q4*1000:.4f} L/s; H = {H4:.4f} m; Re = {Re4:.1f}; lambda = {lam4:.6f}")
print(f"Ph = {Ph4/1000:.4f} kW; Pvr = {Pvr4/1000:.4f} kW; Pel = {Pel4/1000:.4f} kW")
print(f"rad crpke = {hours4:.3f} h/dan; električna energija = {Pel4*hours4/1000:.1f} kWh/dan")
print(f"Energetski rezidual = {history4[-1,2]:.3e} m; Colebrookov = {colebrook_residual(lam4, Re4, eps4/D4):.3e}")
# Objavljeni ručni rezultat (Swamee-Jain): Q = 27,6 L/s i H = 39,1 m.
assert abs(Q4*1000-27.6)<0.05
assert abs(H4-39.1)<0.05
assert abs(history4[-1,2])<1e-6
assert abs(colebrook_residual(lam4, Re4, eps4/D4))<1e-6
assert Re4>4000
assert Ph4<Pvr4<Pel4
assert np.isclose(Pel4*eta_m4*eta_p4, Ph4, rtol=1e-14)
assert abs(hours4-15.1)<0.1
Qrough4, Hrough4, *_ = operating_point4(epsilon=3*eps4)
assert Qrough4<Q4 and Hrough4>H4

## 5. Provjeri krivulje i njihov pomak

Na prvom grafu obje su osi kvantitativne. Drugi prikazuje povijest stvarnog energijskog reziduala. Koeficijent trenja nije unaprijed zamrznut na njegovoj konačnoj vrijednosti. Statička visina ostaje 28 m. Isprekidana krivulja prikazuje ostarjeli vod s trostrukom hrapavošću.

In [ ]:
q_plot4 = np.linspace(0.005,0.045,120)
states4 = np.array([pump_state4(q) for q in q_plot4])
rough_states4 = np.array([pump_state4(q,3*eps4) for q in q_plot4])
fig, axes = plt.subplots(1,2,figsize=(10,4))
axes[0].plot(q_plot4*1000,states4[:,0],label="crpka",color="#1565c0")
axes[0].plot(q_plot4*1000,states4[:,1],label="sustav, $k_s$ = 0,10 mm",color="#c0392b")
axes[0].plot(q_plot4*1000,rough_states4[:,1],"--",label="sustav, trostruka $k_s$",color="#8e44ad")
axes[0].plot(Q4*1000,H4,"o",color="#3a4a56",label="radna točka")
axes[0].set(xlabel="Q (L/s)",ylabel="H (m)",title="Z4: crpka puni vodospremu")
axes[0].legend(fontsize=8)
axes[1].semilogy(history4[:,0],np.maximum(np.abs(history4[:,2]),1e-16),"o-",color="#1e8449")
axes[1].set(xlabel="vanjska iteracija",ylabel=r"$|H_p-H_s|$ (m)",title="Provjera energijske bilance")
for ax in axes:ax.grid(ls=":",alpha=.4)
plt.tight_layout();plt.show()

## 6. Protumači

1. Kako promjena $Re$ za ±10 % utječe na koeficijent trenja u početnom Colebrookovu pokusu? Usporedi s udvostručenjem relativne hrapavosti.
2. Zašto u Z4 treba provjeriti energijski i Colebrookov rezidual? Može li jedan biti malen dok drugi nije?
3. Zašto se pri povećanju hrapavosti radni protok smanjuje, a visina na zadanoj crpkinoj krivulji povećava?
4. Ručni račun u Z4 kreće od $λ=0{,}02$. Koliko bi se razlikovao protok da se stalo nakon prvog koraka? Zašto se Colebrookov rezultat ovdje razlikuje od Swamee–Jainova samo za oko 0,2 %?
5. Zašto najveća od triju snaga pripada električnom ulazu? Koji gubitci nisu sadržani u hidrauličkoj snazi?
6. Može li se nastavni model krivulje crpke koristiti izvan prikazanog raspona protoka bez dodatnih podataka?